In [37]:
library(WGCNA)
# Allow multi-threading
enableWGCNAThreads()
library(DESeq2)
library(TCGAbiolinks)
library(biomaRt)
library(clusterProfiler)
library(org.Hs.eg.db)
library(enrichplot)
library(ggplot2)
library(GEOquery)
library(miRBaseConverter)
library(sva)
library(readr)
library(dplyr)
library(tidyr)
library(EnhancedVolcano)

Allowing parallel execution with up to 11 working processes.


In [38]:
cohort <- "WHITE"
if (cohort == "TAIWAN") {
    config <- list(
        bruh = "hola"
    )
} else if (cohort == "WHITE") {
    config <- list(
        wd = "/home/seba/github_repos/crc_weighted_network/cohort_white/count_matrices_white_byFILENAME",
        rna_merged_counts = "RNA_TMM_T_merged_counts.csv",
        rna_normal_counts = "RNA_TMM_T_normal_counts.csv",
        rna_tumor_counts = "RNA_TMM_T_tumor_counts.csv",
        mirna_merged_counts = "newMIRNA_TMM_T_merged_counts.csv",
        mirna_normal_counts = "newMIRNA_TMM_T_normal_counts.csv",
        mirna_tumor_counts = "newMIRNA_TMM_T_tumor_counts.csv",
        where_to_save = "../random_sampling_matrices"
    )
}

In [39]:
get_random_subset <- function(df, n_size, seed=NULL){
    # 1. Configurar semilla para reproducibilidad (vital para tu tesis)
    if (!is.null(seed)) {
        set.seed(seed)
    }
  
    # 2. Validación de seguridad
    total_rows <- nrow(df)
    
    if (n_size > total_rows) {
        warning(paste("El número solicitado (", n_size, 
                    ") es mayor al total disponible (", total_rows, 
                    "). Devolviendo todas las filas disponibles."))
        n_size <- total_rows
    }
    
    # 3. Muestreo aleatorio
    subset_df <- df %>%
        slice_sample(n = n_size)
    
    return(subset_df)
    }

In [40]:
# set working directory
setwd(config$wd)
rna_tumor <- read.csv(config$rna_tumor_counts, row.names = 1, check.names = FALSE)
mirna_tumor <- read.csv(config$mirna_tumor_counts, row.names = 1, check.names = FALSE)
rna_normal <- read.csv(config$rna_normal_counts, row.names = 1, check.names = FALSE)
mirna_normal <- read.csv(config$mirna_normal_counts, row.names = 1, check.names = FALSE)

In [41]:
rna_size <- nrow(rna_normal)
mirna_size <- nrow(mirna_normal)*3

In [42]:
# do it in a cycle of 10, save a new merged file each time
for (i in 1:10) {
    rna_sample <- get_random_subset(rna_tumor, rna_size, seed=i)
    mirna_sample <- get_random_subset(mirna_tumor, mirna_size, seed=i)
    
    # merge the sample with the normal counts
    rna_merged <- rbind(rna_sample, rna_normal)
    mirna_merged <- rbind(mirna_sample, mirna_normal)
    
    # save the merged file
    write.csv(rna_merged, file = paste0(config$where_to_save, "/", i, "_", config$rna_merged_counts))
    write.csv(mirna_merged, file = paste0(config$where_to_save, "/", i, "_", config$mirna_merged_counts))
}